# Come funziona l'addestramento avversario

Il codice del capitolo [«Come funziona l'addestramento avversario»](https://book.paithon.it/main/GAN/come-funziona.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Come funziona l'addestramento avversario

[Leggi la pagina](https://book.paithon.it/main/GAN/come-funziona.html)


### L'addestramento alternato


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
from torch import nn

# G e D sono due nn.Module, ciascuno con il proprio allenatore
# (opt_G e opt_D): aggiornare l'uno non tocca i pesi dell'altro
criterio = nn.BCEWithLogitsLoss()        # il conto dell'errore (sigmoide dentro)

for epoca in range(n_epoche):
    for batch_reale in loader:
        n = batch_reale.size(0)          # quanti esempi ci sono in questo gruppo
        uni  = torch.ones(n, 1)          # etichette "reale"
        zeri = torch.zeros(n, 1)         # etichette "falso"

        # 1) Passo del discriminatore: distinguere reale da falso
        z = torch.randn(n, dim_rumore)   # rumore
        falsi = G(z).detach()            # campioni sintetici, staccati da G
        loss_D = (criterio(D(batch_reale), uni)   # spinge D(x) -> 1
                  + criterio(D(falsi), zeri))     # spinge D(G(z)) -> 0
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # 2) Passo del generatore: ingannare D (si aggiorna solo G)
        z = torch.randn(n, dim_rumore)
        loss_G = criterio(D(G(z)), uni)  # vuole D(G(z)) -> 1
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()
```


### Che cosa torna indietro


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(8)
D = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 1))
# il dato generato, di cui si vuole il gradiente
x = torch.tensor([0.30, -0.70, 1.20, 0.10], requires_grad=True)
s = D(x)                                                    # il logit
loss_G = F.binary_cross_entropy_with_logits(s, torch.ones(1))  # verso "reale"
loss_G.backward()

print(f"verdetto D(x): {torch.sigmoid(s).item():.4f}")
print("gradiente su x:", " ".join(f"{g:+.4f}" for g in x.grad.tolist()))
x_nuovo = (x - 0.5 * x.grad).detach()                       # mezzo passo
print(f"verdetto dopo mezzo passo: {torch.sigmoid(D(x_nuovo)).item():.4f}")

### La loss non dice niente: come si misura una GAN


In [ ]:

# pt-lento: quattro addestramenti da quattrocento giri, circa un minuto su CPU
import math
import torch
from torch import nn

torch.set_num_threads(1)
angoli = torch.arange(8) * 2 * math.pi / 8
centri = torch.stack([2 * torch.cos(angoli), 2 * torch.sin(angoli)], 1)


def rete(ingressi, uscite):
    return nn.Sequential(nn.Linear(ingressi, 128), nn.LeakyReLU(0.2),
                         nn.Linear(128, 128), nn.LeakyReLU(0.2),
                         nn.Linear(128, uscite))


def giudica(G):
    """Mucchietti coperti (almeno l'1% dei punti) e quota di punti a segno."""
    with torch.no_grad():
        distanze = torch.cdist(G(torch.randn(5000, 2)), centri).min(1)
    a_segno = distanze.values < 0.15
    coperti = sum(bool((a_segno & (distanze.indices == k)).float().mean() >= 0.01)
                  for k in range(8))
    return coperti, a_segno.float().mean().item()


criterio = nn.BCEWithLogitsLoss()
for seme in range(4):
    torch.manual_seed(seme)
    dati = centri[torch.randint(0, 8, (4000,))] + 0.05 * torch.randn(4000, 2)
    G, D = rete(2, 2), rete(2, 1)
    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4)
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4)
    for giro in range(400):
        somme = [0.0, 0.0]
        for batch_reale in dati[torch.randperm(4000)].split(256):
            n = batch_reale.size(0)
            uni, zeri = torch.ones(n, 1), torch.zeros(n, 1)
            falsi = G(torch.randn(n, 2)).detach()
            loss_D = criterio(D(batch_reale), uni) + criterio(D(falsi), zeri)
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()
            loss_G = criterio(D(G(torch.randn(n, 2))), uni)
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()
            somme[0] += loss_D.item() * n; somme[1] += loss_G.item() * n
        if giro in (0, 399):
            coperti, quota = giudica(G)
            print(f"seme {seme}, giro {giro + 1:>3}: loss_D {somme[0] / 4000:.2f}  "
                  f"loss_G {somme[1] / 4000:.2f}  coperti {coperti}/8  "
                  f"a segno {quota:.0%}")

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
m = 50_000
# i dati reali: metà da N(-3, 1) e metà da N(+3, 1)
reali = rng.normal(0, 1, m) + rng.choice([-3.0, 3.0], m)
# il generatore: una gaussiana sola, con la stessa media e la stessa varianza
generati = rng.normal(0, np.sqrt(10), m)


def fid_1d(a, b):
    """Il FID in una dimensione: medie e varianze invece di matrici."""
    va, vb = a.var(), b.var()
    return (a.mean() - b.mean()) ** 2 + va + vb - 2 * np.sqrt(va * vb)


print(f"media e varianza, reali:    {reali.mean():+.3f}  "
      f"{reali.var():.2f}")
print(f"media e varianza, generati: {generati.mean():+.3f}  "
      f"{generati.var():.2f}")
print(f"FID: {fid_1d(reali, generati):.1e}")
print(f"nella fascia |x| < 1: reali {np.mean(np.abs(reali) < 1):.1%}, "
      f"generati {np.mean(np.abs(generati) < 1):.1%}")